In [1]:
import pandas as pd
import sys
sys.path.append(r"C:\Users\Lenovo\OneDrive - Retailstat, LLC\Documents\DataDesk")

from utils.file_handler import read_excel
from config import *

In [7]:
df1 = pd.read_excel(r"C:\Users\Lenovo\OneDrive - Retailstat, LLC\Documents\DataDesk\test\Sample\file-1.xlsx")
df2 = pd.read_excel(r"C:\Users\Lenovo\OneDrive - Retailstat, LLC\Documents\DataDesk\test\Sample\file-2.xlsx")

print(df1.shape)
print(df2.shape)

(1139, 27)
(19, 27)


In [8]:
OUTPUT_COLUMNS = [
    LIST_NAME_COL,
    ALI_COL,
    ADDRESS_COL,
    CITY_COL,
    ZIP_COL,
    POLYGON_STATUS_COL
]

df1_clean = df1[OUTPUT_COLUMNS].copy()
df2_clean = df2[OUTPUT_COLUMNS].copy()

print(df1_clean.shape)
print(df2_clean.shape)

(1139, 6)
(19, 6)


In [9]:
# matched — ALI exists in both
matched = df1_clean[df1_clean[ALI_COL].isin(df2_clean[ALI_COL])]
print("Matched:", len(matched))

# only in file 1
only_file1 = df1_clean[~df1_clean[ALI_COL].isin(df2_clean[ALI_COL])]
print("Only in File1:", len(only_file1))

# only in file 2
only_file2 = df2_clean[~df2_clean[ALI_COL].isin(df1_clean[ALI_COL])]
print("Only in File2:", len(only_file2))

Matched: 19
Only in File1: 1120
Only in File2: 0


In [10]:
# Type 2 — Address Comparison

# matched — Address exists in both
matched_addr = df1_clean[df1_clean[ADDRESS_COL].isin(df2_clean[ADDRESS_COL])]
print("Matched:", len(matched_addr))

# only in file 1
only_file1_addr = df1_clean[~df1_clean[ADDRESS_COL].isin(df2_clean[ADDRESS_COL])]
print("Only in File1:", len(only_file1_addr))

# only in file 2
only_file2_addr = df2_clean[~df2_clean[ADDRESS_COL].isin(df1_clean[ADDRESS_COL])]
print("Only in File2:", len(only_file2_addr))

Matched: 19
Only in File1: 1120
Only in File2: 0


In [11]:
# Type 3 — Migration Comparison

# Step 1 — match both files on Address
merged = df1_clean.merge(
    df2_clean,
    on         = ADDRESS_COL,
    how        = "outer",
    suffixes   = ("_old", "_new"),
    indicator  = True
)

print("Merge complete:", merged.shape)
print(merged["_merge"].value_counts())

Merge complete: (1139, 12)
_merge
left_only     1120
both            19
right_only       0
Name: count, dtype: int64


In [12]:
# only in old file
only_old = merged[merged["_merge"] == "left_only"][[
    LIST_NAME_COL + "_old", ALI_COL + "_old", ADDRESS_COL,
    CITY_COL + "_old", ZIP_COL + "_old", POLYGON_STATUS_COL + "_old"
]]
print("Only in old file:", len(only_old))

# only in new file
only_new = merged[merged["_merge"] == "right_only"][[
    LIST_NAME_COL + "_new", ALI_COL + "_new", ADDRESS_COL,
    CITY_COL + "_new", ZIP_COL + "_new", POLYGON_STATUS_COL + "_new"
]]
print("Only in new file:", len(only_new))

# address matched — now check ALI and Polygon Status
both = merged[merged["_merge"] == "both"]
print("Address matched:", len(both))

Only in old file: 1120
Only in new file: 0
Address matched: 19


In [13]:
# ALI changed
ali_changed = both[both[ALI_COL + "_old"] != both[ALI_COL + "_new"]]
print("ALI changed:", len(ali_changed))

# Polygon lost — old had value, new has None
polygon_lost = both[
    both[POLYGON_STATUS_COL + "_old"].notna() &
    both[POLYGON_STATUS_COL + "_new"].isna()
]
print("Polygon lost:", len(polygon_lost))

# ALI changed AND polygon lost — most critical
ali_changed_polygon_lost = both[
    (both[ALI_COL + "_old"] != both[ALI_COL + "_new"]) &
    (both[POLYGON_STATUS_COL + "_old"].notna()) &
    (both[POLYGON_STATUS_COL + "_new"].isna())
]
print("ALI changed + Polygon lost:", len(ali_changed_polygon_lost))

# ALI changed, never marked in either
ali_changed_never_marked = both[
    (both[ALI_COL + "_old"] != both[ALI_COL + "_new"]) &
    (both[POLYGON_STATUS_COL + "_old"].isna()) &
    (both[POLYGON_STATUS_COL + "_new"].isna())
]
print("ALI changed + Never marked:", len(ali_changed_never_marked))

# clean migration
clean = both[
    (both[ALI_COL + "_old"] == both[ALI_COL + "_new"]) &
    (both[POLYGON_STATUS_COL + "_old"] == both[POLYGON_STATUS_COL + "_new"])
]
print("Clean migration:", len(clean))

ALI changed: 0
Polygon lost: 0
ALI changed + Polygon lost: 0
ALI changed + Never marked: 0
Clean migration: 16
